In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.transform import Rotation

CSV_PATH = "/home/pooja/TATRO/TATRO_Program_files/TATARO_FirmwareV1.0/xarm_ros2/MOCAP_Validation/marker_charuco_tf_log_20260717_153058.csv"
MARKER_HOLD_SAMPLES = 10  # must match marker_hold_samples used when the CSV was recorded

# The four TF chains logged by marker_charuco_tf_logger.py:
#   EEF          - link_base -> link_eef                    (robot forward kinematics)
#   Camera       - link_base -> camera_color_optical_frame   (FK + published hand-eye calibration)
#   CharucoBase  - link_base -> charuco_board_frame           (FK + calibration + live detection)
#   CharucoCam   - camera_color_optical_frame -> charuco_board_frame  (raw detection only)
CHAIN_PREFIXES = ["EEF", "Camera", "CharucoBase", "CharucoCam"]


In [ ]:
df = pd.read_csv(CSV_PATH)
df["Timestamp"] = pd.to_datetime(df["Timestamp"], format="%Y-%m-%d %H:%M:%S.%f")
df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

# .to_numpy() avoids a pandas/matplotlib datetime-indexing incompatibility
# on older matplotlib versions.
ax.plot(df["Timestamp"].to_numpy(), df["User_Marker"].to_numpy(), color="red", linewidth=1.5, label="User_Marker")

# Shade the periods where each TF chain resolved, rather than plotting four
# separate 0/1 lines on top of each other.
valid_colors = {
    "EEF_Valid": "#2a78d6",
    "Camera_Valid": "#1baf7a",
    "CharucoBase_Valid": "#e07b39",
    "CharucoCam_Valid": "#a349a4",
}
for col, color in valid_colors.items():
    valid = df[col].to_numpy().astype(bool)
    ax.fill_between(df["Timestamp"].to_numpy(), 0, 1, where=valid, step="mid",
                    color=color, alpha=0.15, label=col)

ax.set_xlabel("Timestamp")
ax.set_ylabel("Value")
ax.set_title("User_Marker and TF chain validity over time")
ax.legend(loc="upper right", fontsize=8)
fig.autofmt_xdate()

plt.tight_layout()
plt.show()

In [ ]:
# Group consecutive User_Marker == 1 rows into distinct capture events (each
# 'S' key press holds the marker for MARKER_HOLD_SAMPLES samples), and check
# how many samples per event had each TF chain resolve.
marker_idx = np.flatnonzero((df["User_Marker"] == 1).to_numpy())
event_groups = []
run = [marker_idx[0]]
for i in marker_idx[1:]:
    if i == run[-1] + 1:
        run.append(i)
    else:
        event_groups.append(run)
        run = [i]
event_groups.append(run)

# Tag every row with its event_id (-1 for rows outside any event) so later
# cells can group by it directly instead of re-deriving event boundaries.
event_id_col = pd.Series(-1, index=df.index, dtype=int)
for eid, idx in enumerate(event_groups):
    event_id_col.loc[idx] = eid
df["event_id"] = event_id_col

event_rows = []
for event_id, idx in enumerate(event_groups):
    group = df.loc[idx]
    row = {"event_id": event_id, "num_samples": len(group)}
    for prefix in CHAIN_PREFIXES:
        n_valid = int(group[f"{prefix}_Valid"].sum())
        row[f"{prefix}_num_valid"] = n_valid
        row[f"{prefix}_all_valid"] = n_valid == len(group)
    event_rows.append(row)

event_validity_df = pd.DataFrame(event_rows)
print(f"Events found: {len(event_validity_df)}")
for prefix in CHAIN_PREFIXES:
    n_full = int(event_validity_df[f"{prefix}_all_valid"].sum())
    print(f"  {prefix:12s}: fully valid in {n_full}/{len(event_validity_df)} events")

event_validity_df

In [ ]:
# Average pose per event: mean position + chordal-L2 mean orientation
# (Markley's method, via scipy's Rotation.mean()) over the samples of a
# single event, computed separately per chain since a given chain can be
# invalid on a few samples within an otherwise-valid event (e.g. a brief
# ChArUco dropout) without invalidating the whole event.
def average_pose(sub_df, prefix):
    pos = sub_df[[f"{prefix}_X", f"{prefix}_Y", f"{prefix}_Z"]].to_numpy()
    quat = sub_df[[f"{prefix}_QX", f"{prefix}_QY", f"{prefix}_QZ", f"{prefix}_QW"]].to_numpy()
    mean_pos = pos.mean(axis=0)
    mean_quat = Rotation.from_quat(quat).mean().as_quat()
    return mean_pos, mean_quat

event_pose_rows = []
for eid, group in df[df["event_id"] >= 0].groupby("event_id"):
    row = {"event_id": eid}
    for prefix in CHAIN_PREFIXES:
        valid_rows = group[group[f"{prefix}_Valid"] == 1]
        if len(valid_rows) == 0:
            row[f"{prefix}_pos"], row[f"{prefix}_quat"] = None, None
            continue
        pos, quat = average_pose(valid_rows, prefix)
        row[f"{prefix}_pos"], row[f"{prefix}_quat"] = pos, quat
    event_pose_rows.append(row)

event_pose_df = pd.DataFrame(event_pose_rows)
event_pose_df

In [ ]:
# --- Calibration accuracy check --------------------------------------------
# The ChArUco board is physically fixed on the table, so CharucoBase
# (link_base -> charuco_board_frame, i.e. base_T_marker resolved through FK +
# the published hand-eye calibration + the live detection) should be the
# SAME pose across every event, regardless of which robot pose it was
# captured from. Any spread here is real calibration + robot/detection
# error, not a logging artifact - see the TF self-consistency check below,
# which rules out timing/logging bugs as the source.
valid_cb = event_pose_df.dropna(subset=["CharucoBase_pos", "CharucoBase_quat"])
cb_positions = np.stack(valid_cb["CharucoBase_pos"].to_numpy())
cb_quats = np.stack(valid_cb["CharucoBase_quat"].to_numpy())

mean_marker_pos = cb_positions.mean(axis=0)
pos_dev_mm = np.linalg.norm(cb_positions - mean_marker_pos, axis=1) * 1000

mean_marker_rot = Rotation.from_quat(cb_quats).mean()
rot_dev_deg = np.degrees([
    (Rotation.from_quat(q) * mean_marker_rot.inv()).magnitude() for q in cb_quats
])

print(f"Events with valid CharucoBase: {len(cb_positions)} / {len(event_pose_df)}")
print(f"Mean marker position (mm): ({mean_marker_pos[0]*1000:.2f}, {mean_marker_pos[1]*1000:.2f}, {mean_marker_pos[2]*1000:.2f})")
print(f"Position spread  - RMS: {np.sqrt((pos_dev_mm**2).mean()):.2f} mm   max: {pos_dev_mm.max():.2f} mm")
print(f"Orientation spread - RMS: {np.sqrt((rot_dev_deg**2).mean()):.2f} deg   max: {rot_dev_deg.max():.2f} deg")

calibration_check_df = pd.DataFrame({
    "event_id": valid_cb["event_id"].to_numpy(),
    "pos_dev_mm": pos_dev_mm,
    "rot_dev_deg": rot_dev_deg,
})
calibration_check_df

In [ ]:
# --- TF self-consistency check (sanity check on the logger, not the calibration) ---
# CharucoBase should equal Camera composed with CharucoCam almost exactly
# (base_T_marker == base_T_cam @ cam_T_marker), since all four chains come
# from the same TF tree at nearly the same instant. This rules timing/logging
# bugs in/out as an explanation for the spread found above - it should stay
# sub-millimetre regardless of how good or bad the calibration itself is.
def pose_to_H(pos, quat):
    """xyz position + xyzw quaternion -> 4x4 homogeneous transform."""
    H = np.eye(4)
    H[:3, :3] = Rotation.from_quat(quat).as_matrix()
    H[:3, 3] = pos
    return H

all_valid_mask = (df[[f"{p}_Valid" for p in CHAIN_PREFIXES]] == 1).all(axis=1)
valid_marked_df = df[(df["User_Marker"] == 1) & all_valid_mask].reset_index(drop=True)

pos_errs_mm = []
for r in valid_marked_df.itertuples():
    H_cam = pose_to_H([r.Camera_X, r.Camera_Y, r.Camera_Z],
                      [r.Camera_QX, r.Camera_QY, r.Camera_QZ, r.Camera_QW])
    H_cc  = pose_to_H([r.CharucoCam_X, r.CharucoCam_Y, r.CharucoCam_Z],
                      [r.CharucoCam_QX, r.CharucoCam_QY, r.CharucoCam_QZ, r.CharucoCam_QW])
    H_cb  = pose_to_H([r.CharucoBase_X, r.CharucoBase_Y, r.CharucoBase_Z],
                      [r.CharucoBase_QX, r.CharucoBase_QY, r.CharucoBase_QZ, r.CharucoBase_QW])
    H_predicted = H_cam @ H_cc
    pos_errs_mm.append(np.linalg.norm(H_predicted[:3, 3] - H_cb[:3, 3]) * 1000)

pos_errs_mm = np.array(pos_errs_mm)
print(f"Fully-valid marked samples checked: {len(pos_errs_mm)}")
print(f"|CharucoBase - Camera@CharucoCam|  mean: {pos_errs_mm.mean():.3f} mm   max: {pos_errs_mm.max():.3f} mm")
print("Small (sub-mm to low-mm) values here confirm the spread in the cell")
print("above is a real calibration/robot/detection signal, not a TF timing bug.")

In [ ]:
# --- Per-event marker position, visualised ----------------------------------
fig, ax = plt.subplots(figsize=(14, 4))
x = calibration_check_df["event_id"].to_numpy()

ax2 = ax.twinx()
ax.bar(x, calibration_check_df["pos_dev_mm"].to_numpy(), color="#2a78d6", alpha=0.7, label="Position deviation (mm)")
ax2.plot(x, calibration_check_df["rot_dev_deg"].to_numpy(), color="#e07b39", marker="o", label="Orientation deviation (deg)")

ax.set_xlabel("Event ID")
ax.set_ylabel("Position deviation from mean (mm)", color="#2a78d6")
ax2.set_ylabel("Orientation deviation from mean (deg)", color="#e07b39")
ax.set_title("Marker pose deviation per event (should be near zero for a good calibration)")
ax.set_xticks(x)

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc="upper right")

plt.tight_layout()
plt.show()

In [ ]:
# --- Does the error depend on camera pose? -----------------------------
# Test whether pos_dev_mm (from the calibration accuracy check above)
# correlates with camera position, camera-to-marker distance, camera
# orientation deviation from the mean, or individual roll/pitch/yaw - i.e.
# is the error worse at certain viewing angles/ranges (pointing at
# insufficient rotational diversity in the calibration poses, or
# distance-dependent depth/PnP error), or is it closer to a uniform bias
# independent of pose (pointing more at a fixed offset error baked into
# the calibration itself, or non-pose-correlated mechanical backlash)?
#
# Caveat: only 18 events, and several correlations are tested here without
# a multiple-comparison correction - treat anything near p ~ 0.05 as
# suggestive at best, not confirmed. Re-run this cell after collecting a
# larger validation set before trusting any single correlation here.
from scipy.stats import pearsonr

cam_pos = np.stack(event_pose_df["Camera_pos"].to_numpy())
cam_quat = np.stack(event_pose_df["Camera_quat"].to_numpy())
eef_pos = np.stack(event_pose_df["EEF_pos"].to_numpy())

mean_cam_rot = Rotation.from_quat(cam_quat).mean()
cam_orient_dev_deg = np.degrees([
    (Rotation.from_quat(q) * mean_cam_rot.inv()).magnitude() for q in cam_quat
])
cam_to_marker_dist_m = np.linalg.norm(cam_pos - mean_marker_pos, axis=1)
cam_euler_deg = Rotation.from_quat(cam_quat).as_euler("xyz", degrees=True)

candidates = {
    "Camera orientation deviation (deg)": cam_orient_dev_deg,
    "Camera-to-marker distance (m)": cam_to_marker_dist_m,
    "Camera X (m)": cam_pos[:, 0], "Camera Y (m)": cam_pos[:, 1], "Camera Z (m)": cam_pos[:, 2],
    "Camera roll (deg)": cam_euler_deg[:, 0], "Camera pitch (deg)": cam_euler_deg[:, 1], "Camera yaw (deg)": cam_euler_deg[:, 2],
    "EEF X (m)": eef_pos[:, 0], "EEF Y (m)": eef_pos[:, 1], "EEF Z (m)": eef_pos[:, 2],
}

corr_rows = []
for name, values in candidates.items():
    r, p = pearsonr(values, pos_dev_mm)
    corr_rows.append({"quantity": name, "r": r, "p_value": p})
corr_df = pd.DataFrame(corr_rows).sort_values("p_value").reset_index(drop=True)
print(f"n = {len(pos_dev_mm)} events, {len(corr_df)} correlations tested (no multiple-comparison correction)")
corr_df

In [ ]:
# Scatter the two strongest candidates from the table above against
# pos_dev_mm, with the Pearson r/p annotated on each panel.
top2 = corr_df.iloc[:2]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, (_, row) in zip(axes, top2.iterrows()):
    values = candidates[row["quantity"]]
    ax.scatter(values, pos_dev_mm, color="#2a78d6")
    ax.set_xlabel(row["quantity"])
    ax.set_ylabel("Marker position deviation (mm)")
    ax.set_title(f"r={row['r']:.2f}, p={row['p_value']:.3f}")
    ax.grid(color="#e1e0d9", linewidth=1)
    ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

print("No candidate here should be read as a confirmed cause with only 18")
print("events and no multiple-comparison correction - at most a lead to")
print("re-test on a larger validation run.")